# 데이터 분석
담당C : 외국인,국제이동 및 특수집단(다문화,국경이동,특수실태)

- kosis
  - [ ]지방자치단체외국인주민현황(행안부)
  - [ ]출입국자 및 체류외국인통계(법무부)
  - [ ]국제인구이동통계(국가데이터처)
  - [ ]영주귀국사할린동포실태조사(재외동포청)
  - [ ]고령친화용품제조업실태조사(보건산업진흥원)
- 분석 포인트
  - 국경 간 이동, 체류 외국인 및 특정 타겟층 실태 조사 그룹
  - 국가데이터처 외 타 부처(법무부, 행안부, 재외동포청 등)의 행정·실태조사 데이터가 모여 있어, 일반적인 인구통계와 다른 특수 항목(체류자격, 귀국동포, 특정 산업 실태 등)의 예외 케이스를 집중적으로 다루기 좋습니다.


각 카테고리에서 사용자 질의할 수 있는 게 뭔지. 결과 확인해보고 
장르 내 질의, 외 질의 -> LLM 답변 -> 검토

github main에 builder,qa,views를 다운로드 받아서 정해진 카테고리에 대해서 rag 파이프라인 구축해달라고 부탁. 그리고 사용자 질의 및 평가 골든셋 만들어달라고.

## 노트북 구성
RAG는 같은 폴더의 `kosis_views.py`(정제·뷰)와 `kosis_rag_qa.py`(임베딩·검색·답변)를 **불러와서만** 씁니다.
각 단계 끝의 **결과물 보기** 셀에서 그 단계가 만든 파일·데이터(뷰 목록·뷰 차트·인덱스·검색 비교·답변 등)를 확인할 수 있습니다.

| 단계 | 내용 | 출력 (`sample_test/rag_C_sample/`) |
|---|---|---|
| 0 | 설정 | - |
| 1 | 샘플 데이터셋 (표당 상위 20행) | `data/` |
| 2 | 표 둘러보기 | - |
| 3 | 뷰 생성 (`kosis_views`) | `views/` |
| 4 | 임베딩 · RAG 인덱스 (`kosis_rag_qa`) | `index/` |
| 5 | LLM 연결 확인 (질문 1개) | - |
| 6 | 골든셋 작성 — 사람(P) (노트북 또는 `app.py`, 같은 CSV) | `golden_set.csv` |
| 7 | 골든셋 작성 — AI + 정답 검수 + 합치기 | `golden_set.csv`, `golden.jsonl` |
| 8 | 질문을 LLM에 넣어 답변 생성 + 중간 과정(키워드·검색·후보 뷰·최종 선택·참조 링크) | `answers.jsonl` |
| 9 | 정답 vs LLM 답변 비교 + 직접 피드백 기록 | `compare.csv`, `feedback/feedback_<번호>_<날짜시간>.csv` |

## 0. 설정

In [80]:
import argparse
import json
import re
import shutil
import time
from collections import Counter
from html import escape
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML
import nbformat

# 경로
HERE = Path.cwd()                        # exercises/sample_test/
EXERCISES = HERE.parent
SRC_DATA = EXERCISES / "kosis_data"      # 원본 catalog.csv · meta/ · raw/ (읽기만 함)
RAG_ROOT = HERE / "rag_C_sample"         # 이 노트북의 모든 출력
SAMPLE_DATA = RAG_ROOT / "data"          # 샘플 데이터셋 (kosis_views 입력 형식)
VIEWS_OUT = RAG_ROOT / "views"           # 정제 parquet · profiles · records.jsonl
INDEX_DIR = RAG_ROOT / "index"           # BM25·임베딩 인덱스 + 임베딩 캐시

# 실행 설정
SAMPLE_ROWS = 20                         # 표마다 사용할 행 수
VARIANT = "groupC_sample"                # 인덱스 이름

# 담당C 조사
SURVEYS = [
    "지방자치단체외국인주민현황",   # 행안부
    "출입국자및체류외국인통계",     # 법무부
    "국제인구이동통계",             # 국가데이터처
    "영주귀국사할린동포실태조사",   # 재외동포청
    "고령친화용품제조업실태조사",   # 보건산업진흥원
]

load_dotenv(EXERCISES / ".env")          # HCX_API_KEY — kosis_rag_qa import 전에 로드
import kosis_views as kv                 # noqa: E402  정제·뷰 생성
import kosis_rag_qa as rq                # noqa: E402  임베딩·검색·HCX 답변

# kosis_rag_qa 가 정제 데이터·인덱스를 찾을 경로만 지정 (그 외 설정은 두 파일 기본값 그대로)
rq.DATA, rq.VIEWS_DIR, rq.INDEX_DIR = RAG_ROOT, VIEWS_OUT, INDEX_DIR

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)


def show_files(folder, pattern="*", n=8):
    """단계별 결과 폴더 보기: 파일 개수 · 전체 크기 · 앞쪽 n개 목록"""
    folder = Path(folder)
    files = sorted(p for p in folder.rglob(pattern) if p.is_file()) if folder.exists() else []
    print(f"{folder}  →  파일 {len(files)}개 · {sum(p.stat().st_size for p in files) / 1e6:.1f}MB")
    return pd.DataFrame([{"파일": str(p.relative_to(folder)), "KB": round(p.stat().st_size / 1e3, 1)} for p in files[:n]])


print("출력 폴더:", RAG_ROOT)

출력 폴더: c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample


### 0-1. HCX 연결 확인
`kosis_rag_qa.py` 의 설정과 함수를 **그대로** 불러 확인합니다 (여기서 따로 설정하지 않음).
- 설정: `rq.HCX_HOST`(`.env` 의 `HCX_BASE_URL` 에서 호스트만) · `rq.CHAT_MODEL`(`HCX_MODEL`) · `rq.EMBED_PATH`
- 대화: `rq.chat()` — RAG 답변에 쓰는 것과 같은 함수
- 임베딩: `rq._post(rq.EMBED_PATH, …)` 1회 — 캐시에 테스트 문장이 쌓이지 않도록 `embed_one` 대신 직접 호출
- `.env` 를 고쳤다면 커널을 재시작하세요 (`load_dotenv` 는 이미 올라간 값을 덮어쓰지 않고, 호스트·모델은 import 때 읽음)

In [81]:
import os

print("호스트  :", rq.HCX_HOST)
print("대화 모델:", rq.CHAT_MODEL, "| 임베딩 경로:", rq.EMBED_PATH)
print("API 키  :", "있음" if os.getenv("HCX_API_KEY", "").strip() else "없음 (.env 의 HCX_API_KEY 확인)",
      "| 앱 유형:", os.getenv("HCX_APP_TYPE", "testapp"))

HINTS = {401: "인증 실패 → HCX_API_KEY 가 맞는지, 바뀐 키로 커널을 재시작했는지 확인",
         403: "권한 없음 → 키의 앱 유형(HCX_APP_TYPE: testapp/serviceapp)과 모델 사용 권한 확인",
         404: "경로 없음 → HCX_MODEL 이름 또는 HCX_APP_TYPE 확인",
         429: "호출 한도 초과 → 잠시 후 다시 실행"}


def check(name, fn):
    t0 = time.time()
    try:
        result = fn()
        print(f"✅ {name} 성공 ({time.time() - t0:.1f}초): {result}")
    except rq.HcxError as e:
        print(f"❌ {name} 실패: {e}\n   → {HINTS.get(e.http, 'HCX_BASE_URL · 네트워크 상태 확인')}")
    except Exception as e:  # noqa: BLE001  (네트워크 오류 등)
        print(f"❌ {name} 실패: {e.__class__.__name__}: {e}")


check("대화(rq.chat)", lambda: repr(rq.chat(
    [{"role": "user", "content": "연결 확인입니다. '확인'이라고만 답하세요."}], max_tokens=20)))
check("임베딩(rq._post)", lambda: f"{len(rq._post(rq.EMBED_PATH, {'text': '연결 확인'})['embedding'])}차원 벡터")

호스트  : https://clovastudio.stream.ntruss.com
대화 모델: HCX-005 | 임베딩 경로: /v1/api-tools/embedding/v2
API 키  : 있음 | 앱 유형: testapp
✅ 대화(rq.chat) 성공 (0.4초): '확인'
✅ 임베딩(rq._post) 성공 (0.1초): 1024차원 벡터


## 1. 샘플 데이터셋 (표당 상위 20행)
원본 catalog에서 담당C 5개 조사의 선택된 표만 골라 raw는 **상위 `SAMPLE_ROWS` 행만** 복사합니다. 이미 있으면 재사용합니다.

In [82]:
def load_catalog_c(path):
    """catalog.csv → 담당C 조사의 선택된 표만"""
    cat = pd.read_csv(path, dtype=str)
    return cat[cat["category"].isin(SURVEYS) & (cat["selected"] == "True")].reset_index(drop=True)


def table_key(row):
    return f"{row['org_id']}_{row['tbl_id']}"


def raw_file(base, row):
    return base / "raw" / kv.safe(row["category"]) / f"{table_key(row)}.csv"


def meta_file(base, row):
    return base / "meta" / f"{table_key(row)}.json"


def build_sample_dataset(dst=SAMPLE_DATA, n_rows=SAMPLE_ROWS, rebuild=False):
    if (dst / "catalog.csv").exists() and not rebuild:
        print("샘플 데이터셋 재사용:", dst)
        return load_catalog_c(dst / "catalog.csv")
    cat = load_catalog_c(SRC_DATA / "catalog.csv")
    (dst / "meta").mkdir(parents=True, exist_ok=True)
    cat.to_csv(dst / "catalog.csv", index=False, encoding="utf-8-sig")
    for row in cat.to_dict("records"):
        src_raw, src_meta = raw_file(SRC_DATA, row), meta_file(SRC_DATA, row)
        if src_raw.exists():
            out = raw_file(dst, row)
            out.parent.mkdir(parents=True, exist_ok=True)
            pd.read_csv(src_raw, dtype=str, nrows=n_rows, encoding="utf-8-sig").to_csv(out, index=False)
        if src_meta.exists():
            shutil.copy(src_meta, meta_file(dst, row))
    print("샘플 데이터셋 생성:", dst)
    return cat


catalog = build_sample_dataset()
print("표 수:", len(catalog))

샘플 데이터셋 재사용: c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\data
표 수: 254


In [83]:
# 결과물 보기: 샘플 데이터셋 폴더 · catalog · raw 샘플 1개
display(show_files(SAMPLE_DATA, n=5))
display(catalog[["category", "tbl_id", "tbl_nm", "path"]].head())

sample_row = catalog[catalog["category"] == "출입국자및체류외국인통계"].iloc[0]
print("raw 샘플:", raw_file(SAMPLE_DATA, sample_row))
pd.read_csv(raw_file(SAMPLE_DATA, sample_row), dtype=str)

c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\data  →  파일 508개 · 12.9MB


,파일,KB
0,catalog.csv,73.8
1,meta\101_DT_1B28014.json,51.7
2,meta\101_DT_1B28021.json,27.7
3,meta\101_DT_1B28022.json,49.4
4,meta\101_DT_1B28023.json,58.0


,category,tbl_id,tbl_nm,path
0,출입국자및체류외국인통계,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,출입국자및체류외국인통계 > 출입국자통계
1,출입국자및체류외국인통계,DT_091_111_2009_S010,국적(지역) 및 항구별 내외국인 출국자,출입국자및체류외국인통계 > 출입국자통계
2,출입국자및체류외국인통계,DT_091_111_2009_S006,국적(지역) 및 체류자격별 외국인 입국자,출입국자및체류외국인통계 > 출입국자통계
3,출입국자및체류외국인통계,DT_091_111_2009_S005A,국적(지역) 및 월별 외국인 입국자,출입국자및체류외국인통계 > 출입국자통계
4,출입국자및체류외국인통계,DT_091_111_2009_S005,국적(지역) 및 월별 외국인 입국자,출입국자및체류외국인통계 > 출입국자통계


raw 샘플: c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\data\raw\출입국자및체류외국인통계\111_DT_091_111_2009_S009.csv


,ORG_ID,TBL_ID,TBL_NM,C1,C1_OBJ_NM,C1_NM,C2,C2_OBJ_NM,C2_NM,C3,C3_OBJ_NM,C3_NM,ITM_ID,ITM_NM,UNIT_NM,PRD_SE,PRD_DE,DT,LST_CHN_DE
0,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A001,항구별,합계,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,15711643,2021-08-05
1,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A002,항구별,인천공항,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,11706586,2021-08-05
2,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A003,항구별,부산,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,480581,2021-08-05
3,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A004,항구별,인천,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,366984,2021-08-05
4,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A005,항구별,김해,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,1174540,2021-08-05
5,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A006,항구별,제주,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,357238,2021-08-05
6,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A007,항구별,대구,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,44701,2021-08-05
7,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A008,항구별,여수,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,4047,2021-08-05
8,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A009,항구별,광주,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,2,2021-08-05
9,111,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,13102870964A.0,국적(지역)별,총계,13102870964B.0,성별,계,A010,항구별,청주,13103870964T1,국적·지역 및 항구별 내·외국인 입국자,명,A,2009,16621,2021-08-05


## 2. 표 둘러보기
카테고리별 표 구성, 분류축·항목·단위·수록기간, 특수 항목(체류자격·귀화·사할린 등)을 확인해 **물어볼 수 있는 범위**를 파악합니다. 질문 셋(6단계)을 만들 때 참고합니다.

In [84]:
def ok_rows(rows):
    return [r for r in rows or [] if isinstance(r, dict) and "_error" not in r]


def summarize_meta(meta):
    """meta JSON → 표 이름 · 분류(dims) · 항목 · 시점 · 주석"""
    dims, items = {}, []
    for r in ok_rows(meta.get("ITM")):
        if r.get("OBJ_ID") == "ITEM":
            items.append({"id": str(r.get("ITM_ID")), "name": r.get("ITM_NM"), "unit": r.get("UNIT_NM") or ""})
        else:
            dims.setdefault(r.get("OBJ_NM"), []).append(r.get("ITM_NM"))
    tbl = ok_rows(meta.get("TBL"))
    return {"tbl_nm": tbl[0].get("TBL_NM", "") if tbl else "", "dims": dims, "items": items,
            "periods": ok_rows(meta.get("PRD")), "comments": [c.get("CMMT_DC", "") for c in ok_rows(meta.get("CMMT"))]}


def period_range(meta_summary):
    ps = meta_summary["periods"]
    starts = [p["STRT_PRD_DE"] for p in ps if p.get("STRT_PRD_DE")]
    ends = [p["END_PRD_DE"] for p in ps if p.get("END_PRD_DE")]
    return (min(starts) if starts else None, max(ends) if ends else None)


def load_table(row):
    raw, meta = raw_file(SAMPLE_DATA, row), meta_file(SAMPLE_DATA, row)
    return {"row": row,
            "sample": pd.read_csv(raw, dtype=str) if raw.exists() else None,
            "meta": summarize_meta(json.loads(meta.read_text(encoding="utf-8")) if meta.exists() else {})}


tables = [load_table(r) for r in catalog.to_dict("records")]
display(pd.DataFrame([{"category": t["row"]["category"], "has_raw": t["sample"] is not None,
                       "section": t["row"]["section"]} for t in tables])
        .groupby("category").agg(표수=("has_raw", "size"), raw있음=("has_raw", "sum"), 중분류수=("section", "nunique")))

,표수,raw있음,중분류수
category,,,
고령친화용품제조업실태조사,20,20,0
국제인구이동통계,7,7,0
영주귀국사할린동포실태조사,87,87,1
지방자치단체외국인주민현황,84,84,2
출입국자및체류외국인통계,56,55,2


In [85]:
def show_table(category, idx=0):
    """표 하나의 메타 요약 + 샘플 행"""
    t = [t for t in tables if t["row"]["category"] == category][idx]
    m = t["meta"]
    print(f"[{t['row']['tbl_id']}] {t['row']['tbl_nm']}\n경로: {t['row']['path']}")
    print("분류:", {k: len(v) for k, v in m["dims"].items()})
    print("항목:", [(i["name"], i["unit"]) for i in m["items"]][:10])
    print("수록기간:", period_range(m))
    for c in m["comments"][:2]:
        print("주석:", c[:200])
    if t["sample"] is not None:
        display(t["sample"].head(10))


show_table("국제인구이동통계")

[DT_1B28025] 내외국인/성/연령별 국제이동(월간, 연간)
경로: 국제인구이동통계
분류: {'내외국인별': 3, '성별': 3, '연령별': 88}
항목: [('총이동', ''), ('입국자', ''), ('출국자', ''), ('국제순이동', '')]
수록기간: ('2000', '2026.03')
주석: * 국제인구이동통계는 체류기간 90일을 초과하여 상주지를 옮긴 내·외국인 입국자와 출국자를 대상으로 집계합니다.
주석: 


,ORG_ID,TBL_ID,TBL_NM,C1,C1_OBJ_NM,C1_NM,C2,C2_OBJ_NM,C2_NM,C3,C3_OBJ_NM,C3_NM,ITM_ID,ITM_NM,PRD_SE,PRD_DE,DT,LST_CHN_DE
0,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T10,국제순이동,A,2024,125403,2025-06-25
1,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T10,국제순이동,A,2025,74382,2026-07-01
2,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T20,입국자,A,2024,727357,2025-06-25
3,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T20,입국자,A,2025,685242,2026-07-01
4,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T30,출국자,A,2024,601954,2025-06-25
5,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T30,출국자,A,2025,610860,2026-07-01
6,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T40,총이동,A,2024,1329311,2025-06-25
7,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,000,연령별,계,T40,총이동,A,2025,1296102,2026-07-01
8,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,051,연령별,0세,T10,국제순이동,A,2024,-3188,2025-06-25
9,101,DT_1B28025,내외국인/성/연령별 국제이동(월간 연간),0,내외국인별,계,0,성별,계,051,연령별,0세,T10,국제순이동,A,2025,-3380,2026-07-01


In [86]:
# 카테고리 프로파일: 분류축 · 항목 · 단위(샘플의 UNIT_NM) · 수록주기 · 수록기간
def profile(category):
    ts = [t for t in tables if t["row"]["category"] == category]
    dims, items, units, freq = Counter(), Counter(), Counter(), Counter()
    starts, ends = [], []
    for t in ts:
        m = t["meta"]
        dims.update(m["dims"].keys())
        items.update(i["name"] for i in m["items"])
        freq.update(p.get("PRD_SE", "") for p in m["periods"])
        if t["sample"] is not None and "UNIT_NM" in t["sample"]:
            units.update(t["sample"]["UNIT_NM"].dropna().unique())
        s, e = period_range(m)
        starts += [s] if s else []
        ends += [e] if e else []
    top = lambda c, n=5: ", ".join(k for k, _ in c.most_common(n))  # noqa: E731
    return {"category": category, "표수": len(ts), "분류축": top(dims), "항목": top(items), "단위": top(units, 8),
            "수록주기": top(freq), "수록기간": f"{min(starts, default='')}~{max(ends, default='')}"}


pd.DataFrame([profile(c) for c in SURVEYS]).set_index("category")

,표수,분류축,항목,단위,수록주기,수록기간
category,,,,,,
지방자치단체외국인주민현황,84,"행정구역(시군구)별, 국적별, 거주외국인별, 성별, 연령별","외국인, 한국국적을 가지지 않은 자, 유학생, 한국국적, 혼인 귀화자",명,년,2006~2024
출입국자및체류외국인통계,56,"성별, 국적(지역)별, 체류자격별, 국적별, 행정구역(시군구)별","국적 지역 및 체류자격별 체류외국인 현황, 지역 국적 및 성별 등록외국인 현황, 상륙허가자, 항공기수, 승무원수","명, 척, 대, 천원, 건, 개, %","년, 월",1991~2025
국제인구이동통계,7,"성별, 내외국인별, 연령별, 국적별, 체류자격별","총이동, 입국자, 출국자, 국제순이동, 계",명,"년, 월",2000~2026.03
영주귀국사할린동포실태조사,87,"특성별, 한민족 러시아인으로서 소속감과 정체성, 차별받은 경험, 한국 정부 지원 정책 인지도, 기관 및 시설 이용 여부",비율,"명, %, 점",2년,2025~2025
고령친화용품제조업실태조사,20,"특성별, 상황별, 매출액규모별, 종사자규모별, 기업형태별","고령친화 용품 제품 관련 종사자 인력 과부족 상황(사무관리직), 고령친화 용품 제조업 사업체 기업형태, 고령친화 용품 제조업 사업체 유형...","%, 개사, 명, 건, 백만원",년,2021~2024


In [87]:
# 특수 항목(일반 인구통계와 다른 예외 케이스) 키워드 빈도 + 숫자가 아닌 값(통계부호) 확인
SPECIAL = ["체류자격", "국적", "비자", "귀화", "결혼이민", "유학", "불법", "동포", "사할린",
           "귀국", "다문화", "자녀", "고령친화", "제조", "매출", "종사자"]

special, symbols = Counter(), []
for t in tables:
    m, row = t["meta"], t["row"]
    text = " ".join([row["tbl_nm"], *m["dims"].keys(), *[i["name"] for i in m["items"]]])
    special.update((row["category"], k) for k in SPECIAL if k in text)
    if t["sample"] is not None and "DT" in t["sample"]:
        dt = t["sample"]["DT"].astype(str).str.strip()
        bad = dt[pd.to_numeric(dt.str.replace(",", ""), errors="coerce").isna()]
        if len(bad):
            symbols.append({"category": row["category"], "tbl_id": row["tbl_id"], "개수": len(bad),
                            "값": ", ".join(sorted(set(bad))[:5])})

display(pd.Series(special).unstack(fill_value=0))
pd.DataFrame(symbols)

,결혼이민,고령친화,국적,귀국,귀화,다문화,동포,매출,불법,비자,사할린,유학,자녀,제조,종사자,체류자격
고령친화용품제조업실태조사,0,20,0,0,0,0,0,2,0,0,0,0,0,8,10,0
국제인구이동통계,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,2
영주귀국사할린동포실태조사,0,0,1,1,0,0,10,0,0,1,16,0,2,0,0,0
지방자치단체외국인주민현황,3,0,66,0,6,1,3,0,0,0,0,7,15,0,0,0
출입국자및체류외국인통계,2,0,39,0,0,0,1,0,3,0,0,2,0,0,0,18


,category,tbl_id,개수,값
0,지방자치단체외국인주민현황,TX_11025_A008_A,1,*
1,지방자치단체외국인주민현황,TX_11025_A009_A,2,*
2,지방자치단체외국인주민현황,TX_11025_A010_A,1,*
3,지방자치단체외국인주민현황,DT_110025_A037_A,4,*
4,지방자치단체외국인주민현황,DT_110025_A038_A,3,*
5,지방자치단체외국인주민현황,DT_110025_A039_A,3,*
6,지방자치단체외국인주민현황,TX_11025_A017_A,1,*
7,지방자치단체외국인주민현황,TX_11025_A018_A,3,*
8,지방자치단체외국인주민현황,TX_11025_A019_A,2,*
9,지방자치단체외국인주민현황,TX_11025_A021_A,4,"***, -"


## 3. 뷰 생성 (`kosis_views.py`)
- `kv.stage_prepare`: raw → 시점 정렬·숫자화된 parquet + 표 구조(profile)
- `kv.stage_views`: 표 레코드 + 뷰(그래프 1개 단위 단면) 레코드 → `records.jsonl` (RAG 검색 대상)

In [88]:
# force=True: 샘플이 작아 매번 다시 정제해도 수 초 → 결과가 항상 현재 코드 기준
VIEW_ARGS = argparse.Namespace(data=str(SAMPLE_DATA), out=str(VIEWS_OUT), surveys=SURVEYS, tables=None,
                               max_tables=None, force=True, kiwi=False, catalog=None)
kv.stage_prepare(VIEW_ARGS)
kv.stage_views(VIEW_ARGS)

records = rq.load_records(VIEWS_OUT / "records.jsonl")
by_id = {r["id"]: r for r in records}
pd.crosstab(pd.Series([r["category"] for r in records], name="category"),
            pd.Series([r["view_type"] for r in records], name="view_type"), margins=True)

[prepare] 50/254 (완료 49)
[prepare] 100/254 (완료 99)
[prepare] 150/254 (완료 149)
[prepare] 200/254 (완료 199)
[prepare] 250/254 (완료 249)
[prepare] 254/254 (완료 253)
[views] 50/254 표 49, 뷰 94
[views] 100/254 표 99, 뷰 203
[views] 150/254 표 149, 뷰 360
[views] 200/254 표 199, 뷰 480
[views] 250/254 표 249, 뷰 574
[views] 254/254 표 253, 뷰 579
완료: 표 253, 뷰 579 → c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\views


view_type,breakdown,breakdown_trend,items_trend,pyramid,table,trend,All
category,,,,,,,
고령친화용품제조업실태조사,21,21,0,0,20,20,82
국제인구이동통계,15,2,4,0,7,13,41
영주귀국사할린동포실태조사,165,0,0,1,87,0,253
지방자치단체외국인주민현황,125,36,0,1,84,47,293
출입국자및체류외국인통계,53,16,0,3,55,36,163
All,379,75,4,5,253,116,832


### 결과물 보기: 뷰
- `find_views(category=, view_type=, keyword=)`: 만들어진 표·뷰 레코드 목록에서 찾기
- `show_view(id)`: 레코드 하나 자세히 — 설명 · 핵심 수치(facts) · 예시 질문 · 좌표(source) · 실제 데이터(`kv.load_view`) · 차트(`kv.render`)
  - 표 레코드(`table::…`)면 분류·항목·시점 구조와 그 표에서 나온 뷰 목록을 보여줍니다.

In [89]:
display(show_files(VIEWS_OUT, "*.jsonl"))
print("정제 parquet:", len(list((VIEWS_OUT / "processed").glob("*.parquet"))), "개 · 표 구조(profile):",
      len(list((VIEWS_OUT / "profiles").glob("*.json"))), "개")

view_list = pd.DataFrame([{"id": r["id"], "doc_type": r["doc_type"], "view_type": r["view_type"],
                           "category": r["category"], "tbl_id": r["source"]["tbl_id"], "title": r["title"],
                           "facts": len(r.get("facts") or []), "questions": len(r.get("questions") or [])}
                          for r in records])


def find_views(category=None, view_type=None, keyword=None):
    d = view_list
    if category:
        d = d[d["category"] == category]
    if view_type:
        d = d[d["view_type"] == view_type]
    if keyword:
        d = d[d["title"].str.contains(keyword, regex=False)]
    return d.reset_index(drop=True)


find_views(category="출입국자및체류외국인통계").head(15)

c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\views  →  파일 3개 · 8.7MB


,파일,KB
0,records.jsonl,4371.3
1,tables.jsonl,1294.3
2,views.jsonl,3077.0


정제 parquet: 253 개 · 표 구조(profile): 253 개


,id,doc_type,view_type,category,tbl_id,title,facts,questions
0,table::111_DT_091_111_2009_S009,table,table,출입국자및체류외국인통계,DT_091_111_2009_S009,국적(지역) 및 항구별 내외국인 입국자,0,4
1,view::111_DT_091_111_2009_S009::A::00::breakdown,view,breakdown,출입국자및체류외국인통계,DT_091_111_2009_S009,2009 항구별 국적·지역 및 항구별 내·외국인 입국자,6,7
2,table::111_DT_091_111_2009_S010,table,table,출입국자및체류외국인통계,DT_091_111_2009_S010,국적(지역) 및 항구별 내외국인 출국자,0,4
3,view::111_DT_091_111_2009_S010::A::00::breakdown,view,breakdown,출입국자및체류외국인통계,DT_091_111_2009_S010,2011 항구별 국적·지역 및 항구별 내·외국인 출국자,6,7
4,table::111_DT_091_111_2009_S006,table,table,출입국자및체류외국인통계,DT_091_111_2009_S006,국적(지역) 및 체류자격별 외국인 입국자,0,4
5,view::111_DT_091_111_2009_S006::A::00::breakdown,view,breakdown,출입국자및체류외국인통계,DT_091_111_2009_S006,2014 체류자격별 국적·지역 및 체류자격별 외국인 입국자,6,7
6,table::111_DT_091_111_2009_S005A,table,table,출입국자및체류외국인통계,DT_091_111_2009_S005A,국적(지역) 및 월별 외국인 입국자,0,4
7,view::111_DT_091_111_2009_S005A::A::00::trend,view,trend,출입국자및체류외국인통계,DT_091_111_2009_S005A,국적·지역 및 월별 외국인 입국자 추이 (2010~2012),7,6
8,view::111_DT_091_111_2009_S005A::A::01::breakdown,view,breakdown,출입국자및체류외국인통계,DT_091_111_2009_S005A,2012 월별 국적·지역 및 월별 외국인 입국자,6,7
9,view::111_DT_091_111_2009_S005A::A::02::breakdown_trend,view,breakdown_trend,출입국자및체류외국인통계,DT_091_111_2009_S005A,월별 국적·지역 및 월별 외국인 입국자 추이,6,3


In [90]:
def show_view(rec_id, chart=True, rows=20):
    """표·뷰 레코드 하나 자세히 보기"""
    rec = by_id[rec_id]
    print(f"[{rec['doc_type']} · {rec['view_type']}] {rec['title']}\n{rec['description']}")
    if rec.get("questions"):
        print("예시 질문:", rec["questions"][:6])
    if rec.get("aliases"):
        print("동의어:", rec["aliases"][:10])
    if rec.get("caveats"):
        print("주의:", [c[:120] for c in rec["caveats"][:2]])
    print("KOSIS:", rec["source"]["kosis_url"])
    if rec["doc_type"] == "table":                         # 표 레코드: 구조 + 이 표에서 나온 뷰
        prof = json.loads((VIEWS_OUT / "profiles" / f"{rec['source']['table_key']}.json").read_text(encoding="utf-8"))
        display(pd.DataFrame([{"분류": lv["obj_nm"], "종류": lv["kind"], "코드 수": lv["n_codes"],
                               "합계 코드": lv["total_code"], "예시": ", ".join(c["name"] for c in lv["codes"][:6])}
                              for lv in prof["levels"]]))
        print("항목:", [(i["nm"], i["unit"]) for i in prof["items"]][:10])
        print("시점:", {se: f"{ps[0]['label']}~{ps[-1]['label']} ({len(ps)}개)" for se, ps in prof["periods"].items()})
        display(view_list[view_list["id"].isin(rec.get("view_ids", []))][["id", "view_type", "title"]])
        return
    print("핵심 수치:", [f["text"] for f in rec["facts"]])
    print("좌표(source):", {k: rec["source"][k] for k in ("file", "prd_se", "itm_id", "dims", "period")})
    display(kv.load_view(rec["source"], VIEWS_OUT).head(rows))
    if chart:
        fig, _ = kv.render(rec, VIEWS_OUT)
        display(fig)


example_view = find_views(view_type="trend").iloc[0]["id"]
show_view(example_view)

[view · trend] 국적·지역 및 월별 외국인 입국자 추이 (2010~2012)
출입국자및체류외국인통계의 「국적(지역) 및 월별 외국인 입국자」 통계표에서 국적·지역 및 월별 외국인 입국자(명)의 시점별 추이를 2010~2012 기준으로 정리한 자료입니다. 고정 조건: 월별: 계.
예시 질문: ['국적·지역 및 월별 외국인 입국자는 어떻게 변해왔나요?', '국적·지역 및 월별 외국인 입국자 추이', '2012 국적·지역 및 월별 외국인 입국자는 얼마인가요?', '2010년부터 2012까지 국적·지역 및 월별 외국인 입국자 변화', '국적·지역 및 월별 외국인 입국자가 가장 많았던 시기는?', '최근 국적·지역 및 월별 외국인 입국자 증감률은?']
동의어: ['외국 사람', '이주민']
주의: ['승무원 제외(Crew members are excluded)', '행정구역 개편(예: 세종시 신설 2012, 군위군 대구 편입 2023)으로 시점별 지역 범위가 다를 수 있음']
KOSIS: https://kosis.kr/statHtml/statHtml.do?orgId=111&tblId=DT_091_111_2009_S005A
핵심 수치: ['2010 7,774,645명', '2012 10,121,950명', '2010 대비 2012 2,347,305명 증가', '변화율 +30.2%', '최대 2012 10,121,950명', '최소 2010 7,774,645명', '직전 시점(2011) 대비 +16.8%']
좌표(source): {'file': 'processed/111_DT_091_111_2009_S005A.parquet', 'prd_se': 'A', 'itm_id': ['13103870964T1'], 'dims': {'c1': {'codes': ['G001']}, 'c2': {'codes': ['13102870964B.0']}, 'c3': {'codes': ['C01']}}, 'period': {}}


,table_key,prd_se,prd_de,period_label,period_date,c1_code,c1_name,c1_ord,c1_depth,c1_total,c2_code,c2_name,c2_ord,c2_depth,c2_total,c3_code,c3_name,c3_ord,c3_depth,c3_total,itm_id,itm_nm,itm_ord,unit,value,value_raw,symbol
0,111_DT_091_111_2009_S005A,A,2010,2010,2010-01-01,G001,총계,0,0,True,13102870964B.0,계,0,0,True,C01,계,0,0,True,13103870964T1,국적·지역 및 월별 외국인 입국자,0,명,7774645.0,7774645,None
1,111_DT_091_111_2009_S005A,A,2011,2011,2011-01-01,G001,총계,0,0,True,13102870964B.0,계,0,0,True,C01,계,0,0,True,13103870964T1,국적·지역 및 월별 외국인 입국자,0,명,8666503.0,8666503,None
2,111_DT_091_111_2009_S005A,A,2012,2012,2012-01-01,G001,총계,0,0,True,13102870964B.0,계,0,0,True,C01,계,0,0,True,13103870964T1,국적·지역 및 월별 외국인 입국자,0,명,10121950.0,10121950,None


In [91]:
# 위 뷰가 나온 표의 표 레코드 → 분류·항목·시점 구조와 이 표의 뷰 목록
show_view(by_id[example_view]["parent_id"])

[table · table] 국적(지역) 및 월별 외국인 입국자
출입국자및체류외국인통계의 「국적(지역) 및 월별 외국인 입국자」 통계표입니다. 분류: 국적(지역)별(1개: 총계) / 성별(1개: 계) / 월별(7개: 계, 1월, 2월, 3월, 4월, 5월, 6월). 항목: 국적·지역 및 월별 외국인 입국자(명). 수록기간: A 2010~2012 (3개 시점).
예시 질문: ['국적(지역) 및 월별 외국인 입국자 통계', '국적(지역) 및 월별 외국인 입국자 자료 어디서 보나요?', '국적(지역)별 국적·지역 및 월별 외국인 입국자', '성별 국적·지역 및 월별 외국인 입국자']
동의어: ['외국 사람', '이주민', '남녀']
주의: ['승무원 제외(Crew members are excluded)', '행정구역 개편(예: 세종시 신설 2012, 군위군 대구 편입 2023)으로 시점별 지역 범위가 다를 수 있음']
KOSIS: https://kosis.kr/statHtml/statHtml.do?orgId=111&tblId=DT_091_111_2009_S005A


,분류,종류,코드 수,합계 코드,예시
0,국적(지역)별,geo,1,G001,총계
1,성별,sex,1,13102870964B.0,계
2,월별,other,7,C01,"계, 1월, 2월, 3월, 4월, 5월"


항목: [('국적·지역 및 월별 외국인 입국자', '명')]
시점: {'A': '2010~2012 (3개)'}


,id,view_type,title
7,view::111_DT_091_111_2009_S005A::A::00::trend,trend,국적·지역 및 월별 외국인 입국자 추이 (2010~2012)
8,view::111_DT_091_111_2009_S005A::A::01::breakdown,breakdown,2012 월별 국적·지역 및 월별 외국인 입국자
9,view::111_DT_091_111_2009_S005A::A::02::breakdown_trend,breakdown_trend,월별 국적·지역 및 월별 외국인 입국자 추이


## 4. 임베딩 · RAG 인덱스 (`kosis_rag_qa.py`)
`rq.stage_index` 로 BM25 + HCX 임베딩 인덱스를 만들고 `rq.Index` 로 불러옵니다.
레코드 수가 바뀌면 다시 만들며, 임베딩은 캐시되어 새 텍스트만 호출합니다.
- `idx.search_bm25()`: BM25 단독 / `idx.search_dense()`: 임베딩 단독 / `idx.search()`: 하이브리드(RRF)

In [92]:
index_meta = INDEX_DIR / VARIANT / "meta.json"
if not index_meta.exists() or json.loads(index_meta.read_text(encoding="utf-8"))["n"] != len(records):
    rq.stage_index(argparse.Namespace(records=VIEWS_OUT / "records.jsonl", doc_type=None, limit=None,
                                      variant=VARIANT, dense_mode="single", workers=4))
idx = rq.Index(VARIANT)

In [93]:
# 결과물 보기: 인덱스 정보 · 파일 · 임베딩 캐시
print({k: idx.meta[k] for k in ("variant", "n", "dense_mode", "embed_model", "dim", "n_vectors", "built")})
display(show_files(INDEX_DIR))
print("임베딩 캐시:", rq.cache().db.execute("SELECT count(*) FROM emb").fetchone()[0], "건")

{'variant': 'groupC_sample', 'n': 832, 'dense_mode': 'single', 'embed_model': 'clova-embedding-v2', 'dim': 1024, 'n_vectors': 832, 'built': '2026-10-27T09:57:44'}
c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\index  →  파일 5개 · 10.5MB


,파일,KB
0,embed_cache.sqlite,6713.3
1,groupC_sample\bm25.pkl,336.3
2,groupC_sample\dense.npy,3408.0
3,groupC_sample\dense_owner.npy,3.5
4,groupC_sample\meta.json,88.5


임베딩 캐시: 1437 건


In [94]:
def compare_search(query, k=5):
    """같은 질문을 BM25 단독 · 임베딩 단독 · 하이브리드로 검색해 상위 k개 제목을 나란히 비교"""
    results = {"BM25": idx.search_bm25(query, k=k), "임베딩": idx.search_dense(query, k=k),
               "하이브리드": idx.search(query, k=k)}
    return pd.DataFrame({name: [h["title"] for h in hits] + [""] * (k - len(hits)) for name, hits in results.items()},
                        index=range(1, k + 1))


compare_search("체류자격별 체류외국인 현황")

,BM25,임베딩,하이브리드
1,2025 체류자격별 체류자격별 불법체류외국인 현황,2009 체류자격별 국적·지역 및 체류자격별 체류외국인 현황,2012 체류자격별 국적 지역 및 체류자격별 체류외국인 현황
2,2012 체류자격별 국적 지역 및 체류자격별 체류외국인 현황,2013 체류자격별 국적 지역 및 체류자격별 체류외국인 현황,2012 체류자격별 국적 지역 및 체류자격별 체류외국인 현황
3,2012 체류자격별 국적 지역 및 체류자격별 체류외국인 현황,2010 체류자격별 국적 지역 및 체류자격별 체류외국인 현황,2013 체류자격별 국적 지역 및 체류자격별 체류외국인 현황
4,체류자격별 불법체류외국인 현황 추이 (2018~2025),2012 체류자격별 국적 지역 및 체류자격별 체류외국인 현황,2010 체류자격별 국적 지역 및 체류자격별 체류외국인 현황
5,체류자격별 체류자격별 불법체류외국인 현황 추이,2012 체류자격별 국적 지역 및 체류자격별 체류외국인 현황,2009 체류자격별 국적·지역 및 체류자격별 체류외국인 현황


## 5. LLM 연결 확인
1. HCX 대화 API 응답 확인 (`rq.chat`)
2. 질문 1개를 RAG로 답변 (`rq.answer`) → `show_trace()` 로 **답변 생성 중간 과정** 확인

`show_trace(답변)` 이 보여주는 과정 (8단계 질문 셋에도 같은 함수 사용)

| 순서 | 내용 | 출처 |
|---|---|---|
| ① 키워드 · 검색어 | 질문의 BM25 키워드, HCX가 재작성한 검색어 · 하위 질문 | `rq.tokenize`, 답변의 `rewritten` · `sub_questions` |
| ② 검색어별 검색 | 검색어마다 RAG 검색 상위 문서 (BM25 순위 · 임베딩 순위 · 유사도) | `idx.search` 로 다시 검색 (임베딩 캐시 사용) |
| ③ 최종 후보 | 검색어별 결과를 합친(RRF) 후보 | 답변의 `retrieved` |
| ④ 후보 뷰 · 만든 데이터 | LLM에 넘긴 후보 뷰(최대 4개)와 질문 조건(지역·연도·성별) 적용 결과, 그 조건으로 다시 조회한 데이터 | 답변의 `conditions`, `rq.apply_conditions` + `kv.load_view` |
| ⑤ 최종 선택 | LLM이 근거로 쓴 뷰(`used_ids`) · 그래프 뷰(`chart_id`) | 답변 |
| ⑥ 답변 | 답변 · 핵심 수치 · 확인 불가 여부 · 자료에서 확인 안 된 숫자 · 주의 | 답변 |
| ⑦ 참조 링크 | 근거 표 KOSIS 링크 · 출처 | 답변의 `used_ids` · `sources` |

※ 뷰 자체는 3단계에서 미리 만들어 두고, 답변할 때는 검색으로 후보 뷰를 고른 뒤 **질문 조건을 좌표에 적용해 질문 전용 데이터**를 만듭니다 (④).

In [95]:
# 표 id → 표 이름 · KOSIS 링크 (5·9단계에서 근거 표 표시에 사용)
TABLE_INFO = {r["source"]["tbl_id"]: {"name": r["tbl_nm"], "url": r["source"]["kosis_url"]}
              for r in records if r["doc_type"] == "table"}


def used_tables(ans):
    """답변 근거(used_ids) → 표 id 목록 (순서 유지, 중복 제거)"""
    return list(dict.fromkeys(by_id[u]["source"]["tbl_id"] for u in ans.get("used_ids") or [] if u in by_id))


def table_link(ref):
    """표 id → 표 이름 링크 (인덱스에 없는 표의 URL이면 URL 그대로 링크)"""
    info = TABLE_INFO.get(ref)
    if info:
        return f'<a href="{info["url"]}" target="_blank">{escape(info["name"])}</a>'
    ref = str(ref)
    return f'<a href="{escape(ref)}" target="_blank">{escape(ref)}</a>' if ref.startswith("http") else escape(ref)


def table_text(ref):
    info = TABLE_INFO.get(ref)
    return f"{info['name']} {info['url']}" if info else str(ref)


t0 = time.time()
reply = rq.chat([{"role": "user", "content": "연결 확인입니다. '확인'이라고만 답하세요."}], max_tokens=20)
print(f"HCX 응답: {reply!r} ({time.time() - t0:.1f}초)")

HCX 응답: '확인' (0.3초)


In [96]:
TEST_QUESTION = "2025년 국적별 외국인 입국자 수가 가장 많은 나라는?"

res = rq.answer(TEST_QUESTION, VARIANT, save=False, idx=idx)
print(f"답변 완료 ({res['elapsed_s']}초) — 아래 셀에서 중간 과정 확인")

답변 완료 (4.95초) — 아래 셀에서 중간 과정 확인


In [97]:
def _section(title):
    display(HTML(f"<h4 style='margin:14px 0 4px'>{title}</h4>"))


def _hits_table(hits, score_key="score"):
    return pd.DataFrame([{"순위": i, "종류": h.get("doc_type"), "제목": h["title"], "점수": h.get(score_key),
                          "BM25 순위": h.get("bm25_rank"), "임베딩 순위": h.get("dense_rank"),
                          "임베딩 유사도": h.get("dense_sim"), "id": h["id"]} for i, h in enumerate(hits, 1)])


def question_data(rec, question, rows):
    """후보 뷰 좌표에 질문 조건을 적용해 다시 조회한 데이터 — LLM에 넘어가는 것과 같은 열(시점·분류명·항목·값·단위).
    조건이 없으면 뷰 미리보기"""
    try:
        src, applied, _ = rq.apply_conditions(rec, question)
        if applied:
            return rq.condition_rows(src, rows)[0].head(rows)
    except Exception as e:  # noqa: BLE001
        print("  (조건 적용 데이터 조회 실패:", e, ")")
    pv = rec.get("preview") or {}
    return pd.DataFrame(pv.get("rows") or [], columns=pv.get("columns") or None).head(rows)


def show_trace(ans, search_k=5, data_rows=8, chart=False):
    """rq.answer 결과 1건의 중간 과정: 키워드 → 검색 → 최종 후보 → 후보 뷰·만든 데이터 → 최종 선택 → 답변 → 참조 링크"""
    q = ans["question"]
    missing = [k for k in ("rewritten", "retrieved", "conditions") if k not in ans]
    display(HTML(f"<h3 style='margin:8px 0'>❓ {escape(q)}</h3>"))
    if missing:
        print(f"⚠ 저장된 답변에 {missing} 정보가 없습니다 → answers.jsonl 에서 이 질문을 지우고 8단계를 다시 실행하세요.")

    _section("① 키워드 · 검색어")
    queries = list(dict.fromkeys([ans.get("rewritten") or q] + list(ans.get("sub_questions") or [])))
    print("질문 키워드(BM25 토큰):", rq.tokenize(q))
    print("재작성 검색어:", queries[0], "→ 키워드", rq.tokenize(queries[0]))
    for s in queries[1:]:
        print("하위 질문:", s, "→ 키워드", rq.tokenize(s))

    _section(f"② 검색어별 RAG 검색 결과 (상위 {search_k})")
    for sq in dict.fromkeys(queries + [q]):
        display(HTML(f"<b>검색어:</b> {escape(sq)}{' (원 질문, 가중치 0.5)' if sq == q and sq not in queries else ''}"))
        display(_hits_table(idx.search(sq, k=search_k)))

    _section("③ 검색어별 결과를 합친 최종 후보 (RRF)")
    if ans.get("retrieved"):
        display(_hits_table(ans["retrieved"], score_key="fused"))

    _section("④ LLM에 넘긴 후보 뷰 · 질문 조건 적용으로 만든 데이터")
    used, chart_id = set(ans.get("used_ids") or []), ans.get("chart_id")
    for rid, cond in (ans.get("conditions") or {}).items():
        rec = by_id.get(rid)
        if rec is None:
            continue
        mark = ("✅ 근거" if rid in used else "") + (" 📈 그래프" if rid == chart_id else "")
        display(HTML(f"<b>{escape(rec['title'])}</b> <small>({rec['view_type']} · {escape(rid)})</small> {mark}<br>"
                     f"표: {table_link(rec['source']['tbl_id'])}<br>"
                     f"적용한 조건: {escape(str(cond.get('applied') or '-'))} · "
                     f"적용 못 한 조건: {escape(str(cond.get('missing') or '-'))}"))
        display(question_data(rec, q, data_rows))

    _section("⑤ 최종 선택")
    for rid in ans.get("used_ids") or []:
        print("근거 뷰:", rid, "|", by_id[rid]["title"] if rid in by_id else "")
    print("그래프 뷰:", chart_id or "-")
    if chart and chart_id in by_id:
        fig, _ = kv.render(by_id[chart_id], VIEWS_OUT)
        display(fig)

    _section("⑥ 답변")
    print(ans.get("answer") or "")
    print("\n핵심 수치:", ans.get("key_points") or [])
    print("확인 불가:", bool(ans.get("insufficient")), "| 질문 조건 충족:", ans.get("condition_matched"),
          "| 자료에서 확인 안 된 숫자:", ans.get("unverified_numbers") or [])
    for c in ans.get("caveats") or []:
        print("주의:", c)

    _section("⑦ 참조 링크")
    display(HTML("<br>".join(map(table_link, used_tables(ans))) or "-"))
    for s in ans.get("sources") or []:
        print("출처:", s)


show_trace(res, chart=True)

질문 키워드(BM25 토큰): ['2025', '년', '국적', '외국인', '입국자', '수', '많', '나라']
재작성 검색어: 2025년 국적별 외국인 입국자 수가 가장 많은 나라는? → 키워드 ['2025', '년', '국적', '외국인', '입국자', '수', '많', '나라']
하위 질문: 2025년 외국인 입국자 중 국적이 가장 많은 나라 → 키워드 ['2025', '년', '외국인', '입국자', '중', '국적', '많', '나라']


,순위,종류,제목,점수,BM25 순위,임베딩 순위,임베딩 유사도,id
0,1,view,입국자 추이 (2024~2025),0.03652,1,0,0.6301,view::101_DT_1B28025::A::01::trend
1,2,view,2025 연령별 입국자,0.03458,7,2,0.6078,view::101_DT_1B28025::A::06::breakdown
2,3,view,입국자 추이 (2014~2025),0.03252,0,1,0.6224,view::101_DT_1B28022::A::00::trend
3,4,view,인원 추이 (2018~2025),0.02949,38,4,0.5996,view::111_DT_1B040A36::A::00::trend
4,5,view,명수 추이 (2019~2025),0.02904,32,9,0.5702,view::111_DT_1B040A35_1::A::00::trend


,순위,종류,제목,점수,BM25 순위,임베딩 순위,임베딩 유사도,id
0,1,view,입국자 추이 (2024~2025),0.03652,1,0,0.6356,view::101_DT_1B28025::A::01::trend
1,2,view,2025 연령별 입국자,0.03416,9,2,0.6290,view::101_DT_1B28025::A::06::breakdown
2,3,view,입국자 추이 (2014~2025),0.03252,0,1,0.6331,view::101_DT_1B28022::A::00::trend
3,4,view,인원 추이 (2018~2025),0.02905,40,5,0.6004,view::111_DT_1B040A36::A::00::trend
4,5,view,2010 연령별 국적·지역 및 연령별 외국인 입국자(승무원 제외),0.02838,8,11,0.5858,view::111_DT_091_111_2009_S007::A::01::breakdown


,순위,종류,제목,점수,BM25 순위,임베딩 순위,임베딩 유사도,id
0,1,view,입국자 추이 (2024~2025),0.03279,1,0,0.6301,view::101_DT_1B28025::A::01::trend
1,2,view,2025 연령별 입국자,0.03226,7,2,0.6078,view::101_DT_1B28025::A::06::breakdown
2,3,view,입국자 추이 (2014~2025),0.03175,0,1,0.6224,view::101_DT_1B28022::A::00::trend
3,4,view,인원 추이 (2018~2025),0.03125,38,4,0.5996,view::111_DT_1B040A36::A::00::trend
4,5,view,국적·지역 및 월별 외국인 입국자 추이 (2010~2012),0.03008,2,22,0.5533,view::111_DT_091_111_2009_S005A::A::00::trend


,period_label,c1_name,c2_name,c3_name,itm_nm,value,unit
0,2025,계,계,계,입국자,685242.0,


...

,period_label,c1_name,c2_name,itm_nm,value,unit


,period_label,c1_name,itm_nm,value,unit
0,2025,계,인원,357598.0,명


근거 뷰: view::101_DT_1B28025::A::01::trend | 입국자 추이 (2024~2025)
그래프 뷰: view::101_DT_1B28025::A::01::trend


답변: 2025년에 국적별 외국인 입국자가 가장 많은 나라는 중국입니다. 자료에 따르면 2025년 중국의 외국인 입국자는 약 357,598명으로 다른 국가의 입국자보다 많습니다.

summary: 2025년 외국인 입국자가 가장 많은 국가는 중국이며, 해당 인원은 약 357,598명입니다.
key_points: 2025년, 중국, 외국인 입국자, 약 357,598명
used_ids: view::111_DT_1B040A36::A::00::trend
chart_id: view::111_DT_1B040A36::A::00::trend

핵심 수치: []
확인 불가: False | 질문 조건 충족: True | 자료에서 확인 안 된 숫자: []
주의: * 국제인구이동통계는 체류기간 90일을 초과하여 상주지를 옮긴 내·외국인 입국자와 출국자를 대상으로 집계합니다. * 국제인구이동통계는 체류기간에 관계없이 모든 출입국 건수를 집계하는 출입국자통계(법무부)와는 집계대상에 차이가 있습니다. * 월간은 보도자료없이 kosis로 통계표만 서비스합니다. * 월간은 잠정 데이터이며, 연간 공표(7월)후에 확정됩니다. …
주의: 전수/표본 구분 표기가 없는 표


출처: 자료: 「국제인구이동통계」 (KOSIS) https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1B28025


## 6. 골든셋 작성 — 사람(P)
골든셋은 **노트북(6·7단계)** 과 **Streamlit 앱(`app.py`)** 어디서 작성해도 같은 파일 `rag_C_sample/golden_set.csv` 에 저장됩니다.
두 곳 모두 `golden_store.py` 의 같은 규칙을 씁니다.
- id 자동 부여 (P001… / AI001…), 이미 있는 질문은 건너뜀 → **셀을 다시 실행해도 중복 저장되지 않음**
- 링크에서 표 id 자동 추출 (로그인 주소처럼 인코딩된 링크 포함), 표가 RAG 인덱스에 없으면 경고
- 사람(P) 문항은 자동으로 정답 검수 완료(`reviewed = y`)

```
streamlit run why/kosis_study/exercises/sample_test/app.py
```
- 열: `id` · `author`(P 사람 / AI) · `reviewed`(정답 검수 완료 y / 검수 전 n) · `question` · `answer` · `in_rag`(y/n) · `link` · `tbl_ids` · `note` · `created_at`
- 앱이 켜져 있으면 노트북에서 저장한 뒤 앱 화면을 새로고침하면 보입니다. 채점(good/bad · 엣지포인트)은 앱의 ② 골든셋 채점 화면에서 합니다.

In [98]:
import importlib

import golden_store as gs                 # 앱과 같은 골든셋 CSV 도구
gs = importlib.reload(gs)                 # golden_store.py 를 고쳐도 커널 재시작 없이 반영

GOLDEN_CSV = RAG_ROOT / "golden_set.csv"


def add_golden(items, author):
    """골든셋 추가 → golden_set.csv. items: dict 목록 또는 JSON/노트북 형식 문자열.
    항목에 "author" 가 있으면 그 값, 없으면 author 사용. 이미 있는 질문은 건너뜀"""
    if isinstance(items, str):
        items = gs.parse_golden_text(items) if items.strip() else []
    if not items:
        return
    df = gs.load_golden(GOLDEN_CSV)
    rows, warnings = gs.make_rows(items, df, author, TABLE_INFO.keys())
    dup = [w for w in warnings if "이미 있는 질문" in w]
    for w in warnings:
        if w not in dup:
            print("⚠", w)
    if rows:
        gs.save_golden(pd.concat([df, pd.DataFrame(rows)], ignore_index=True), GOLDEN_CSV)
        print(f"✅ {len(rows)}개 저장: {', '.join(r['id'] for r in rows)}")
    if dup:
        print(f"(이미 저장된 질문 {len(dup)}개는 건너뜀)")


def golden_summary():
    df = gs.load_golden(GOLDEN_CSV)
    ai = df[df["author"] == "AI"]
    print(f"{GOLDEN_CSV.name} — 전체 {len(df)}개 · P {int((df['author'] == 'P').sum())} · AI {len(ai)} "
          f"(AI 정답 검수 ✅ {int((ai['reviewed'] == 'y').sum())} / ⏳ {int((ai['reviewed'] != 'y').sum())})")
    return df

In [99]:
# 사람(P)이 만든 골든셋 — 항목을 추가하고 셀 실행 (question 필수, 나머지 선택)
NEW_GOLDEN_P = [
    # {"question": "2025년 사할린동포는 총 몇 명인가요?",
    #  "answer": "2,434명", "in_rag": "y",
    #  "link": "https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A002_2025",
    #  "note": "연령 표, 특성별 전체 사례수"},
]

add_golden(NEW_GOLDEN_P, author="P")
golden_csv = golden_summary()
golden_csv[golden_csv["author"] == "P"]

golden_set.csv — 전체 19개 · P 3 · AI 16 (AI 정답 검수 ✅ 16 / ⏳ 0)


,id,author,reviewed,question,answer,in_rag,link,tbl_ids,note,created_at
0,P001,P,y,2025년 사할린동포는 총 몇 명인가요?,"2,434명",y,https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A002_2025&co...,DT_182001_A002_2025,,2026-10-27 08:35:11
1,P002,P,y,2023년 사할린동포는 총 몇 명인가요?,2023년 자료는 존재하지 않습니다.,y,https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A002_2025&co...,DT_182001_A002_2025,,2026-10-27 08:35:11
2,P003,P,y,1세대 사할린동포는 총 몇 명인가요?,확인 불가 (샘플에는 1세대 사례수 없음. 원본 1251명),n,https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A002_2025&co...,DT_182001_A002_2025,,2026-10-27 08:35:11


## 7. 골든셋 작성 — AI + 정답 검수 + 합치기
1. **AI 골든셋 추가**: `NEW_GOLDEN_AI` 에 dict 목록을 넣거나, `NEW_GOLDEN_AI_TEXT` 에 AI가 만든 JSON을 그대로 붙여넣기 (둘 다 가능)
2. **정답 수정 · 검수**: `EDITS` 에 고칠 문항과 열만 적기. 검수를 마친 AI 문항은 `"reviewed": "y"`
   - 바뀐 값만 저장하고 무엇이 바뀌었는지 출력합니다. 앱에서 같은 문항을 나중에 고쳤다면, 노트북 `EDITS` 를 다시 실행할 때 앱의 수정이 덮어써지니 **적용한 항목은 지우거나 주석 처리**하세요.
3. **합치기**: 사람 · AI 문항을 합쳐 8·9단계에서 쓸 `golden` 을 만듭니다 (qid = CSV의 `id`).

In [100]:
# AI(클로드 등)가 만든 골든셋 — dict 목록 또는 JSON 붙여넣기 (author 가 없으면 AI, reviewed 는 기본 검수 전)
NEW_GOLDEN_AI = [
    # {"question": "...", "answer": "...", "in_rag": "y", "link": "https://kosis.kr/...tblId=..."},
]
NEW_GOLDEN_AI_TEXT = """
"""

add_golden(NEW_GOLDEN_AI, author="AI")
add_golden(NEW_GOLDEN_AI_TEXT, author="AI")
golden_csv = golden_summary()
golden_csv[golden_csv["author"] == "AI"]

golden_set.csv — 전체 19개 · P 3 · AI 16 (AI 정답 검수 ✅ 16 / ⏳ 0)


,id,author,reviewed,question,answer,in_rag,link,tbl_ids,note,created_at
3,AI001,AI,y,2009년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,"11,706,586명",y,https://kosis.kr/statHtml/statHtml.do?orgId=111&tblId=DT_091_111_2009_S009,DT_091_111_2009_S009,"국적(지역) 총계 · 성별 계 · 항구별 인천공항, 2009년",2026-10-27 08:35:11
4,AI002,AI,y,2025년 영주귀국 사할린동포 중 80대의 비율은 얼마인가요?,46.3%,y,https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A002_2025,DT_182001_A002_2025,"연령 표, 특성별 전체(사례수 2,434명) 기준",2026-10-27 08:35:11
5,AI003,AI,y,2024년 고령친화 용품 제조업체 중 개인사업체의 비율은 얼마인가요?,53.4%,y,https://kosis.kr/statHtml/statHtml.do?orgId=358&tblId=DT_358005_001,DT_358005_001,"기업형태 표, 특성별 전체(사례수 1,096개사) 기준. 회사법인 46.1%, 회사외법인 0.5%",2026-10-27 08:35:11
6,AI004,AI,y,2025년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,확인 불가 (샘플 데이터에는 2009년만 있음),n,https://kosis.kr/statHtml/statHtml.do?orgId=111&tblId=DT_091_111_2009_S009,DT_091_111_2009_S009,"표는 RAG에 있지만 상위 20행이 모두 2009년. 원본 전체 데이터 값은 35,095,017명",2026-10-27 08:35:11
7,AI005,AI,y,외국인 근로자의 평균 월급은 얼마인가요?,확인 불가 (담당C 조사에 임금 자료 없음),n,,,주제는 외국인이지만 임금 통계는 담당C 5개 조사에 없음 (경계 질문),2026-10-27 08:35:11
8,AI006,AI,y,2024년 합계출산율은 얼마인가요?,확인 불가 (담당 범위 밖),n,,,인구동향조사 영역 (장르 외 질문),2026-10-27 08:35:11
9,AI007,AI,y,2024년 우리나라 외국인주민은 모두 몇 명인가요?,"2,583,626명",y,https://kosis.kr/statHtml/statHtml.do?orgId=110&tblId=TX_11025_A000_A,TX_11025_A000_A,"시도별 외국인주민 현황, 행정구역·거주외국인·성별 모두 합계",2026-10-27 10:56:26
10,AI008,AI,y,외국인주민 수는 2015년에서 2024년 사이에 몇 명 늘었나요?,"872,613명 증가 (1,711,013명 → 2,583,626명)",y,https://kosis.kr/statHtml/statHtml.do?orgId=110&tblId=TX_11025_A000_A,TX_11025_A000_A,두 시점 값의 차이를 계산해야 하는 질문,2026-10-27 10:56:26
11,AI009,AI,y,2024년 외국인주민 중 남자는 몇 명인가요?,"1,367,083명",y,https://kosis.kr/statHtml/statHtml.do?orgId=110&tblId=TX_11025_A000_A,TX_11025_A000_A,"성별 남자, 행정구역 합계",2026-10-27 10:56:26
12,AI010,AI,y,2024년 중국(한국계) 외국인 근로자는 몇 명인가요?,"68,273명",y,https://kosis.kr/statHtml/statHtml.do?orgId=110&tblId=TX_11025_A005_A,TX_11025_A005_A,"한국국적을 가지지 않은 자-외국인 근로자(계), 시군구 합계. 전체 외국인 근로자는 502,634명. 국적 조건 적용 여부 확인용",2026-10-27 10:56:26


In [101]:
# 정답 수정 · 검수 — 고칠 문항 id 와 바꿀 열만 적기 (answer · in_rag · link · note · reviewed · question · author)
EDITS = {
    # "AI001": {"answer": "11,706,586명", "reviewed": "y"},
    # "AI004": {"reviewed": "y"},                          # 정답이 맞으면 검수 완료만
}

df = gs.load_golden(GOLDEN_CSV)
changes = []
for item_id, fields in EDITS.items():
    if item_id not in set(df["id"]):
        print(f"⚠ 없는 id: {item_id}")
        continue
    before = df.set_index("id").loc[item_id]
    df = gs.update_item(df, item_id, **fields)
    after = df.set_index("id").loc[item_id]
    changes += [f"{item_id} {col}: {before[col]!r} → {after[col]!r}" for col in gs.EDITABLE if before[col] != after[col]]
if changes:
    gs.save_golden(df, GOLDEN_CSV)
    print("✅ 저장\n  " + "\n  ".join(changes))
elif EDITS:
    print("바뀐 값 없음 (이미 반영됨)")

golden_csv = golden_summary()
pending = golden_csv[golden_csv["reviewed"] != "y"]
print("⏳ 정답 검수 전:", ", ".join(pending["id"]) or "없음")
pending[["id", "question", "answer", "in_rag", "link", "note"]]

golden_set.csv — 전체 19개 · P 3 · AI 16 (AI 정답 검수 ✅ 16 / ⏳ 0)
⏳ 정답 검수 전: 없음


,id,question,answer,in_rag,link,note


In [102]:
# 6·7단계 합치기 → golden (8·9단계에서 사용)
golden_csv = gs.load_golden(GOLDEN_CSV)            # 위에서 저장한 최신 내용
GOLDEN_COLS = ["qid", "source", "question", "gold_answer", "gold_tbl_ids", "gold_refs", "in_rag", "category", "note"]


def gold_refs(link):
    """표시용 정답 근거: 인덱스에 있는 표는 표 id, 표 id를 못 읽었거나 인덱스에 없으면 원래 URL"""
    refs = []
    for url in gs.parse_links(link):
        ids = [t for t in gs.tbl_ids_from_links([url]) if t in TABLE_INFO]
        refs += ids or [url]
    return list(dict.fromkeys(refs))



def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")


golden = pd.DataFrame({
    "qid": golden_csv["id"], "source": golden_csv["author"], "question": golden_csv["question"],
    "gold_answer": golden_csv["answer"], "gold_tbl_ids": golden_csv["tbl_ids"].str.split(),
    "gold_refs": golden_csv["link"].map(gold_refs), "in_rag": golden_csv["in_rag"],
    "category": "", "note": golden_csv["note"]}, columns=GOLDEN_COLS)
write_jsonl(RAG_ROOT / "golden.jsonl", golden.to_dict("records"))

outside = sorted({t for ids in golden["gold_tbl_ids"] for t in ids if t not in TABLE_INFO})
if outside:
    print("⚠ 링크의 표가 RAG 인덱스(샘플 데이터)에 없음:", outside)
print(f"골든셋 {len(golden)}개 · 정답 작성 {int((golden['gold_answer'] != '').sum())}개 → {RAG_ROOT / 'golden.jsonl'}")
golden.drop(columns=["gold_refs"])

골든셋 19개 · 정답 작성 19개 → c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\golden.jsonl


,qid,source,question,gold_answer,gold_tbl_ids,in_rag,category,note
0,P001,P,2025년 사할린동포는 총 몇 명인가요?,"2,434명",[DT_182001_A002_2025],y,,
1,P002,P,2023년 사할린동포는 총 몇 명인가요?,2023년 자료는 존재하지 않습니다.,[DT_182001_A002_2025],y,,
2,P003,P,1세대 사할린동포는 총 몇 명인가요?,확인 불가 (샘플에는 1세대 사례수 없음. 원본 1251명),[DT_182001_A002_2025],n,,
3,AI001,AI,2009년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,"11,706,586명",[DT_091_111_2009_S009],y,,"국적(지역) 총계 · 성별 계 · 항구별 인천공항, 2009년"
4,AI002,AI,2025년 영주귀국 사할린동포 중 80대의 비율은 얼마인가요?,46.3%,[DT_182001_A002_2025],y,,"연령 표, 특성별 전체(사례수 2,434명) 기준"
5,AI003,AI,2024년 고령친화 용품 제조업체 중 개인사업체의 비율은 얼마인가요?,53.4%,[DT_358005_001],y,,"기업형태 표, 특성별 전체(사례수 1,096개사) 기준. 회사법인 46.1%, 회사외법인 0.5%"
6,AI004,AI,2025년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,확인 불가 (샘플 데이터에는 2009년만 있음),[DT_091_111_2009_S009],n,,"표는 RAG에 있지만 상위 20행이 모두 2009년. 원본 전체 데이터 값은 35,095,017명"
7,AI005,AI,외국인 근로자의 평균 월급은 얼마인가요?,확인 불가 (담당C 조사에 임금 자료 없음),[],n,,주제는 외국인이지만 임금 통계는 담당C 5개 조사에 없음 (경계 질문)
8,AI006,AI,2024년 합계출산율은 얼마인가요?,확인 불가 (담당 범위 밖),[],n,,인구동향조사 영역 (장르 외 질문)
9,AI007,AI,2024년 우리나라 외국인주민은 모두 몇 명인가요?,"2,583,626명",[TX_11025_A000_A],y,,"시도별 외국인주민 현황, 행정구역·거주외국인·성별 모두 합계"


## 8. 질문을 LLM에 넣어 답변 생성
질문 셋의 **질문만** `rq.answer` 에 넘깁니다 (정답은 넘기지 않음).
답변은 질문마다 `answers.jsonl` 에 바로 저장되어, 다시 실행하면 새 질문만 호출합니다 (질문 문장이 같으면 재사용).

In [103]:
ANSWERS_PATH = RAG_ROOT / "answers.jsonl"
# 답변 + 중간 과정(검색어·검색 결과·후보 뷰 조건·최종 선택·출처)까지 저장 → 8-1 show_trace 에서 사용
ANSWER_KEYS = ("answer", "summary", "key_points", "rewritten", "sub_questions", "retrieved", "conditions",
               "condition_matched", "used_ids", "chart_id", "sources", "insufficient", "unverified_numbers",
               "caveats", "elapsed_s")


def load_answers(path=ANSWERS_PATH):
    if not path.exists():
        return {}
    return {d["question"]: d for d in map(json.loads, path.open(encoding="utf-8"))}


def generate_answers(question_list, path=ANSWERS_PATH):
    answers = load_answers(path)
    todo = [q for q in dict.fromkeys(question_list) if q not in answers]
    print(f"답변 대상 {len(todo)} / 질문 {len(set(question_list))} (저장된 답변 {len(answers)})")
    with path.open("a", encoding="utf-8") as f:
        for i, q in enumerate(todo, 1):
            try:
                res = rq.answer(q, VARIANT, save=False, idx=idx)
                ans = {k: res.get(k) for k in ANSWER_KEYS}
            except Exception as e:  # noqa: BLE001  (1건 실패로 전체를 멈추지 않음)
                ans = {"answer": f"[오류] {e.__class__.__name__}: {e}", "error": True}
            answers[q] = {"question": q, **ans}
            f.write(json.dumps(answers[q], ensure_ascii=False, default=str) + "\n")
            print(f"  {i}/{len(todo)} {q[:50]}", flush=True)
    return answers


answers = generate_answers(golden["question"].tolist())

답변 대상 0 / 질문 19 (저장된 답변 19)


In [104]:
# 결과물 보기: 질문별 답변 요약 (전문은 9단계 비교표)
print(ANSWERS_PATH)
pd.DataFrame([{"qid": g["qid"], "질문": g["question"], "답변(앞부분)": str(a.get("answer") or "")[:80],
               "확인 불가": bool(a.get("insufficient")), "근거 표 수": len(used_tables(a)),
               "확인 안 된 숫자": ", ".join(a.get("unverified_numbers") or []), "초": a.get("elapsed_s")}
              for g in golden.to_dict("records") if (a := answers.get(g["question"]))],
             columns=["qid", "질문", "답변(앞부분)", "확인 불가", "근거 표 수", "확인 안 된 숫자", "초"])

c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\answers.jsonl


,qid,질문,답변(앞부분),확인 불가,근거 표 수,확인 안 된 숫자,초
0,P001,2025년 사할린동포는 총 몇 명인가요?,"답변: 2025년 사할린 동포 수는 약 2,434명입니다.\n\nkey points: 2,434명",False,1,,80.42
1,P002,2023년 사할린동포는 총 몇 명인가요?,"answer: 현재 제공된 자료들은 모두 2025년을 기준으로 작성되었으며, 2023년의 사할린 동포 수에 대한 직접적인 정보는 없습니다...",True,1,,6.87
2,P003,1세대 사할린동포는 총 몇 명인가요?,답변: 2025년 기준으로 1세대 사할린 동포는 약 51.4명입니다. 이 정보는 「영주귀국사할린동포실태조사」 자료를 바탕으로 합니다. [...,False,1,,3.68
3,AI001,2009년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,"답변: 2009년 인천공항으로 들어온 내·외국인 입국자는 **11,706,586명**입니다. \n- 자료: 「출입국자및체류외국인통계」 (...",True,1,,3.38
4,AI002,2025년 영주귀국 사할린동포 중 80대의 비율은 얼마인가요?,답변: 2025년 영주귀국 사할린동포 중 80대의 비율은 **46.30**%입니다.\n\nused_ids: view::182_DT_182...,False,1,,3.49
5,AI003,2024년 고령친화 용품 제조업체 중 개인사업체의 비율은 얼마인가요?,답변: 2024년 고령친화 용품 제조업에서 개인사업체가 차지하는 비율은 **53.40**%입니다.\n\nkey_points:\n- 202...,False,1,,4.82
6,AI004,2025년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,"summary: 2025년 인천공항으로 들어온 내·외국인 입국자의 수는 약 685,242명입니다.\n\nkey_points: 2025년 ...",True,1,,6.24
7,AI005,외국인 근로자의 평균 월급은 얼마인가요?,answer: 제공된 자료에는 외국인 근로자의 평균 월급 정보가 포함되어 있지 않습니다. 이 자료들은 주로 국적별 외국인 근로자의 수를 ...,False,1,,5.11
8,AI006,2024년 합계출산율은 얼마인가요?,answer: 죄송하지만 본 자료에는 2024년 합계출산율에 대한 정보가 없습니다. 해당 정보는 통계청이나 관련 기관의 출산 및 인구 관...,False,1,,4.67
9,AI007,2024년 우리나라 외국인주민은 모두 몇 명인가요?,"답변: 2024년 우리나라 외국인 주민 수는 약 492,000명으로 추정됩니다.\n\n- 동북아시아 외국인: 276,568 + 123,9...",False,1,"492,000, 553,136, 565,872, 63,998",7.69


### 8-1. 답변 생성 중간 과정
질문마다 `show_trace()` 로 ① 키워드 · 검색어 → ② 검색어별 검색 → ③ 최종 후보 → ④ 후보 뷰 · 만든 데이터 → ⑤ 최종 선택 → ⑥ 답변 → ⑦ 참조 링크를 보여줍니다.
- `TRACE_QIDS = ["P001", "AI002"]` 처럼 일부만 볼 수 있고, `None` 이면 전체
- 예전에 저장된 답변에는 중간 과정이 없을 수 있습니다 → `answers.jsonl` 에서 해당 질문을 지우고 8단계 재실행

In [105]:
TRACE_QIDS = None          # 예: ["P001"] — None 이면 전체

for g in golden.to_dict("records"):
    if TRACE_QIDS and g["qid"] not in TRACE_QIDS:
        continue
    ans = answers.get(g["question"])
    display(HTML(f"<hr><h2 style='margin:6px 0'>{g['qid']}</h2>"))
    if ans is None:
        print("답변 없음 (8단계 실행 필요)")
        continue
    show_trace(ans)

질문 키워드(BM25 토큰): ['2025', '년', '사할린', '동포', '명']
재작성 검색어: 2025년 사할린동포는 총 몇 명인가요? → 키워드 ['2025', '년', '사할린', '동포', '명']


,순위,종류,제목,점수,BM25 순위,임베딩 순위,임베딩 유사도,id
0,1,view,2025 사할린동포 수별 비율,0.03679,0,0,0.6929,view::182_DT_182001_A038_2025::A::01::breakdown
1,2,view,2025 사할린동포 수별 비율,0.03626,1,1,0.6882,view::182_DT_182001_A036_2025::A::01::breakdown
2,3,view,2025 사할린동포 수별 비율,0.03575,2,2,0.6863,view::182_DT_182001_A037_2025::A::01::breakdown
3,4,view,2025 사할린동포 수별 비율,0.03525,3,3,0.6774,view::182_DT_182001_A039_2025::A::01::breakdown
4,5,view,2025 사할린동포 세대별 비율,0.03308,12,4,0.6691,view::182_DT_182001_A004_2025::A::01::breakdown


,순위,종류,제목,점수,BM25 순위,임베딩 순위,임베딩 유사도,id
0,1,view,2025 사할린동포 수별 비율,0.01639,0,0,0.6929,view::182_DT_182001_A038_2025::A::01::breakdown
1,2,view,2025 사할린동포 수별 비율,0.01613,1,1,0.6882,view::182_DT_182001_A036_2025::A::01::breakdown
2,3,view,2025 사할린동포 수별 비율,0.01587,2,2,0.6863,view::182_DT_182001_A037_2025::A::01::breakdown
3,4,view,2025 사할린동포 수별 비율,0.01562,3,3,0.6774,view::182_DT_182001_A039_2025::A::01::breakdown
4,5,view,2025 사할린동포 세대별 비율,0.01538,12,4,0.6691,view::182_DT_182001_A004_2025::A::01::breakdown


,period_label,c1_name,c2_name,itm_nm,value,unit
0,2025,전체,사례수,비율,2434.0,명
1,2025,전체,전혀 없다,비율,7.6,%
2,2025,전체,1~2명,비율,37.4,%
3,2025,전체,3~5명,비율,24.4,%
4,2025,전체,6명 이상,비율,30.6,%


,period_label,c1_name,c2_name,itm_nm,value,unit
0,2025,전체,사례수,비율,2434.0,명
1,2025,전체,전혀 없다,비율,1.8,%
2,2025,전체,1~2명,비율,14.2,%
3,2025,전체,3~5명,비율,18.0,%
4,2025,전체,6명 이상,비율,66.1,%


...

,c2_name,value
0,결혼이민자,158620.0
1,귀화자 등,151002.0
2,귀화·인지 및 외국인,9750.0
3,국내출생,189411.0
4,내국인,253958.0
5,외국인,48752.0


,c2_name,value
0,한국인배우자,151681.0
1,결혼이민자 및 귀화자 등,309622.0
2,자녀,199161.0
3,기타동거인,302710.0


근거 뷰: view::110_TX_11025_A016_A::A::00::breakdown | 2024 국적별 외국인
그래프 뷰: view::110_TX_11025_A016_A::A::00::breakdown


answer: 현재 제공된 자료에는 2024년의 다문화 가구 수에 대한 정보가 포함되어 있지 않습니다. 대신 2016년도 자료를 통해 당시 다문화 가구 수를 알려드릴 수는 있습니다.

summary: 2024년 다문화 가구 수에 대한 정보는 제공되지 않았습니다.

key_points:
- 2016년 결혼 이민자 및 귀화자는 약 309,622명이었습니다.
- 2016년 자녀는 약 199,161명이었습니다.
used_ids: []
chart_id: []

핵심 수치: []
확인 불가: False | 질문 조건 충족: True | 자료에서 확인 안 된 숫자: []
주의: 매년 11월1일 기준 *은 5인 미만인 경우임 기타 외국인 : 기업투자, 취재 등 체류자격이 외국인근로자ㆍ결혼이민자ㆍ유학생ㆍ외국국적동포에 해당하지 않는 자 및 단기체류기간 도과 불법체류자(지역별ㆍ국적별 비율에 따라 시ㆍ도, 시ㆍ군ㆍ구 현황에 안분)
주의: 전수/표본 구분 표기가 없는 표


출처: 자료: 「지방자치단체외국인주민현황」 (KOSIS) https://kosis.kr/statHtml/statHtml.do?orgId=110&tblId=TX_11025_A016_A


## 9. 정답 vs LLM 답변 비교 + 피드백
### 9-1. 비교표
문항마다 **질문 / RAG에 있음(y·n) / 정답 근거 표(링크) / 정답 / LLM 답변 / 답변 근거 표(링크)** 를 나란히 보여줍니다.
- `표 일치`: 답변 근거 표에 정답 표가 있으면 ✅, 다른 표만 썼으면 ❌, 비교할 수 없으면 `-`
- `숫자 힌트`: 정답에 있는 숫자 중 답변에도 있는 개수 (판단 보조용, 채점 아님)

In [106]:
NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")


def numbers(text):
    return {n.replace(",", "") for n in NUM_RE.findall(str(text))}


def number_hint(gold, answer):
    g = numbers(gold)
    return f"{len(g & numbers(answer))}/{len(g)}" if g else ""


rows = []
for g in golden.to_dict("records"):
    ans = answers.get(g["question"], {})
    used = used_tables(ans)
    hit = "-" if not used or not g["gold_tbl_ids"] else ("✅" if set(used) & set(g["gold_tbl_ids"]) else "❌")
    rows.append({"qid": g["qid"], "RAG에 있음": g["in_rag"], "질문": g["question"],
                 "정답 근거 표": g["gold_refs"], "정답": g["gold_answer"],
                 "LLM 답변": str(ans.get("answer") or ""), "확인 불가": "예" if ans.get("insufficient") else "",
                 "답변 근거 표": used, "표 일치": hit, "숫자 힌트": number_hint(g["gold_answer"], ans.get("answer"))})
compare = pd.DataFrame(rows, columns=["qid", "RAG에 있음", "질문", "정답 근거 표", "정답", "LLM 답변", "확인 불가",
                                      "답변 근거 표", "표 일치", "숫자 힌트"])

# CSV: 근거 표는 "표 이름 URL"
link_cols = ["정답 근거 표", "답변 근거 표"]
compare.assign(**{c: compare[c].map(lambda ids: " | ".join(map(table_text, ids))) for c in link_cols}) \
       .to_csv(RAG_ROOT / "compare.csv", index=False, encoding="utf-8-sig")

# 화면: 링크 클릭 가능, 긴 글 줄바꿈
view = compare.copy()
for c in link_cols:
    view[c] = view[c].map(lambda ids: "<br>".join(map(table_link, ids)) or "-")
for c in ["질문", "정답", "LLM 답변"]:
    view[c] = view[c].map(lambda s: escape(s).replace("\n", "<br>"))
style = ("<style>.cmp td,.cmp th{text-align:left;vertical-align:top;padding:6px;border:1px solid #ccc}"
         ".cmp td:nth-child(3),.cmp td:nth-child(6){min-width:260px}</style>")
print(f"문항 {len(compare)}개 | RAG에 있음 y {int((compare['RAG에 있음'] == 'y').sum())} · "
      f"n {int((compare['RAG에 있음'] == 'n').sum())} · 정답 미작성 {int((compare['RAG에 있음'] == '').sum())}")
display(HTML(style + view.to_html(escape=False, index=False, classes="cmp")))

문항 19개 | RAG에 있음 y 12 · n 7 · 정답 미작성 0


qid,RAG에 있음,질문,정답 근거 표,정답,LLM 답변,확인 불가,답변 근거 표,표 일치,숫자 힌트
P001,y,2025년 사할린동포는 총 몇 명인가요?,연령,"2,434명","답변: 2025년 사할린 동포 수는 약 2,434명입니다.key points: 2,434명",,사할린동포 수 - 한국 생활 도움 주는 사람,❌,1/1
P002,y,2023년 사할린동포는 총 몇 명인가요?,연령,2023년 자료는 존재하지 않습니다.,"answer: 현재 제공된 자료들은 모두 2025년을 기준으로 작성되었으며, 2023년의 사할린 동포 수에 대한 직접적인 정보는 없습니다. 따라서 2023년 사할린 동포의 총 수를 정확히 알 수는 없습니다.summary: 2023년 사할린동포 수에 관한 직접적인 자료가 없어 답변할 수 없습니다.key_points: - 모든 자료는 2025년 기준임- 2023년에 해당하는 자료 없음used_ids: view::182_DT_182001_A038_2025::A::01::breakdown, view::182_DT_182001_A036_2025::A::01::breakdown, view::182_DT_182001_A037_2025::A::01::breakdown, view::182_DT_182001_A039_2025::A::01::breakdownchart_id: view::182_DT_182001_A038_2025::A::01::breakdown",예,사할린동포 수 - 한국 생활 도움 주는 사람,❌,1/1
P003,n,1세대 사할린동포는 총 몇 명인가요?,연령,확인 불가 (샘플에는 1세대 사례수 없음. 원본 1251명),답변: 2025년 기준으로 1세대 사할린 동포는 약 51.4명입니다. 이 정보는 「영주귀국사할린동포실태조사」 자료를 바탕으로 합니다. [used_ids: 'view::182_DT_182001_A004_2025::A::01::breakdown'],,사할린동포 세대,❌,1/2
AI001,y,2009년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,국적(지역) 및 항구별 내외국인 입국자,"11,706,586명","답변: 2009년 인천공항으로 들어온 내·외국인 입국자는 **11,706,586명**입니다. - 자료: 「출입국자및체류외국인통계」 (KOSIS), 2009년 기준",예,국적(지역) 및 항구별 내외국인 입국자,✅,1/1
AI002,y,2025년 영주귀국 사할린동포 중 80대의 비율은 얼마인가요?,연령,46.3%,답변: 2025년 영주귀국 사할린동포 중 80대의 비율은 **46.30**%입니다.used_ids: view::182_DT_182001_A002_2025::A::01::breakdown,,연령,✅,0/1
AI003,y,2024년 고령친화 용품 제조업체 중 개인사업체의 비율은 얼마인가요?,고령친화 용품 제조업 사업체 기업형태,53.4%,"답변: 2024년 고령친화 용품 제조업에서 개인사업체가 차지하는 비율은 **53.40**%입니다.key_points:- 2024년 고령친화 용품 제조업 사업체 기업 형태 중 개인사업체 비율: 53.40%- 전체 고령친화 용품 제조업 사업체 수: 1,096개사",,고령친화 용품 제조업 사업체 기업형태,✅,0/1
AI004,n,2025년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?,국적(지역) 및 항구별 내외국인 입국자,확인 불가 (샘플 데이터에는 2009년만 있음),"summary: 2025년 인천공항으로 들어온 내·외국인 입국자의 수는 약 685,242명입니다.key_points: 2025년 입국자 수 | 약 685,242명used_ids: view::101_DT_1B28025::A::01::trendanswer: 2025년에 인천공항으로 들어온 내·외국인 입국자의 수는 약 685,242명으로 나타났습니다. 이는 국제인구이동통계를 기반으로 하여 2024년부터 2025년까지의 입국자 추이를 나타낸 자료에서 확인할 수 있으며, 해당 자료에서는 연령별 구분을 하지 않고 전체 입국자에 대한 정보만을 제공하고 있습니다. 따라서, 인천공항을 통해 입국한 내·외국인의 총 수를 의미합니다. 단위는 제공되지 않았으나, 일반적으로 인구 관련 통계에서 사용되는 '명' 단위로 추정할 수 있습니다.",예,"내외국인/성/연령별 국제이동(월간, 연간)",❌,0/1
AI005,n,외국인 근로자의 평균 월급은 얼마인가요?,-,확인 불가 (담당C 조사에 임금 자료 없음),answer: 제공된 자료에는 외국인 근로자의 평균 월급 정보가 포함되어 있지 않습니다. 이 자료들은 주로 국적별 외국인 근로자의 수를 나타내고 있습니다.summary: 외국인 근로자의 평균 월급 정보는 제공되지 않았습니다.key_points: - 외국인 근로자의 국적별 분포 정보만 포함됨 - 평균 월급 관련 데이터 없음used_ids: - view::110_TX_11025_A006_A::A::01::breakdown- view::110_TX_11025_A005_A::A::00::breakdown- view::110_TX_11025_A006_A::A::00::breakdown- view::110_TX_11025_A005_A::A::01::breakdownchart_id: 해당 없음,,한국국적을 가지지 않은 자-외국인 근로자(남),-,
AI006,n,2024년 합계출산율은 얼마인가요?,-,확인 불가 (담당 범위 밖),answer: 죄송하지만 본 자료에는 2024년 합계출산율에 대한 정보가 없습니다. 해당 정보는 통계청이나 관련 기관의 출산 및 인구 관련 통계 자료를 통해 확인하실 수 있습니다.summary: 제공된 자료에서는 2024년 합계출산율에 대한 정보를 찾을 수 없습니다.key_points: - 2024년 합계출산율 정보 없음- 관련 기관의 출산 및 인구 통계 자료 참고 필요used_ids: [],,시군구별 유형별 외국인주민 자녀,-,
AI007,y,2024년 우리나라 외국인주민은 모두 몇 명인가요?,시도별 외국인주민 현황,"2,583,626명","답변: 2024년 우리나라 외국인 주민 수는 약 492,000명으로 추정됩니다.- 동북아시아 외국인: 276,568 + 123,917 + 152,651 = 553,136명- 동남아시아 외국인: 282,936 + 142,117 + 140,819 = 565,872명- 서남아시아 외국인: 31,954 + 12,643 + 19,311 = 63,998명- 위 세 지역의 외국인 수를 합산하여 대략적인 총 외국인 수를 구할 수 있습니다.단, 제공된 자료에는 성별이나 특정 조건을 만족하는 외국인의 수가 포함되어 있으므로 실제 전체 외국인 수는 이보다 많을 가능성이 큽니다. 또한, 매년 11월 1일 기준으로 집계된 수치이므로 시기별로 변동이 있을 수 있음을 유의해야 합니다. [used_ids: view::110_TX_11025_A014_A::A::00::breakdown, view::110_TX_11025_A016_A::A::00::breakdown, view::110_TX_11025_A015_A::A::00::breakdown]",,한국국적을 가지지 않은 자-기타 외국인(계),❌,0/1


### 9-2. 피드백 기록
비교표를 보고 qid별로 판단과 메모를 적습니다. 셀을 실행할 때마다 비교표와 함께 `feedback/feedback_<번호>_<날짜시간>.csv` 로 **새 파일**이 저장됩니다 (이전 기록은 그대로 남음).
- 판단: `"O"`(맞음) / `"X"`(틀림) / `"△"`(부분적으로 맞음)

In [107]:
FEEDBACK = {
    # "P001": ("O", "정답과 일치, 근거 표도 맞음"),
}

FEEDBACK_DIR = RAG_ROOT / "feedback"


def next_feedback_path(folder=FEEDBACK_DIR):
    """feedback_<번호>_<날짜시간>.csv — 번호는 기존 파일의 가장 큰 번호 + 1"""
    folder.mkdir(parents=True, exist_ok=True)
    nums = [int(m.group(1)) for p in folder.glob("feedback_*.csv") if (m := re.match(r"feedback_(\d+)_", p.name))]
    return folder / f"feedback_{max(nums, default=0) + 1:03d}_{time.strftime('%Y%m%d_%H%M%S')}.csv"


feedback = compare.drop(columns=["답변 근거 표", "정답 근거 표"]).assign(
    판단=compare["qid"].map(lambda q: FEEDBACK.get(q, ("", ""))[0]),
    메모=compare["qid"].map(lambda q: FEEDBACK.get(q, ("", ""))[1]))
feedback_path = next_feedback_path()
feedback.to_csv(feedback_path, index=False, encoding="utf-8-sig")

done = feedback[feedback["판단"] != ""]
print(f"피드백 {len(done)}/{len(feedback)}개 작성 → {feedback_path}")
display(show_files(FEEDBACK_DIR, "feedback_*.csv", n=20))                # 지금까지 저장된 피드백 기록
if len(done):
    display(pd.crosstab(done["RAG에 있음"], done["판단"], margins=True))
feedback[feedback["판단"] == ""][["qid", "질문"]]       # 아직 피드백이 없는 문항

피드백 0/19개 작성 → c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\feedback\feedback_003_20261027_115258.csv
c:\dev\t4core-stat-agent\why\kosis_study\exercises\sample_test\rag_C_sample\feedback  →  파일 3개 · 0.0MB


,파일,KB
0,feedback_001_20261027_095951.csv,4.7
1,feedback_002_20261027_104942.csv,4.7
2,feedback_003_20261027_115258.csv,10.3


,qid,질문
0,P001,2025년 사할린동포는 총 몇 명인가요?
1,P002,2023년 사할린동포는 총 몇 명인가요?
2,P003,1세대 사할린동포는 총 몇 명인가요?
3,AI001,2009년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?
4,AI002,2025년 영주귀국 사할린동포 중 80대의 비율은 얼마인가요?
5,AI003,2024년 고령친화 용품 제조업체 중 개인사업체의 비율은 얼마인가요?
6,AI004,2025년 인천공항으로 들어온 내·외국인 입국자는 몇 명인가요?
7,AI005,외국인 근로자의 평균 월급은 얼마인가요?
8,AI006,2024년 합계출산율은 얼마인가요?
9,AI007,2024년 우리나라 외국인주민은 모두 몇 명인가요?


---
## 부록. RAG가 어려워할 표 2개 살펴보기 + 질문 만들기
샘플 데이터(표당 20행)에서 RAG가 틀리기 쉬운 표 2개를 골라 20행을 그대로 보고, 표마다 **RAG에 있는 질문(y)** 과 **없는 질문(n)** 을 1개씩 만들었습니다.

| 표 | 어려운 이유 |
|---|---|
| 「결혼 여부」 (영주귀국사할린동포실태조사, `DT_182001_A028_2025`) | 사할린 실태조사 표 87개가 모두 `특성별 × 응답` 구조이고 사례수 2,434명이 똑같이 반복 → 검색에서 비슷한 표(배우자 출신 민족 등)와 헷갈리기 쉬움. 응답 항목 이름이 문장처럼 길어 질문 표현과 잘 안 맞음. 샘플 20행에는 특성별 `전체·남성·여성` 만 있음 |
| 「국적(지역) 및 체류자격별 체류외국인 현황」 (출입국자및체류외국인통계, `DT_1B040A5`) | **같은 이름의 표가 4개** — 2009(`A5C`) · 2010~2011(`A5B`) · 2012(`A5`) · 2013~2025(`A5A`) → 연도를 보고 맞는 표를 골라야 함. 국적 · 체류자격 조건은 RAG의 조건 적용 대상이 아님. 샘플 20행이 전부 `가나 · 여자 · 2012` 이고 총계 행이 없음 |

In [108]:
# 두 표의 샘플 데이터(20행) 그대로 보기
HARD_TABLES = ["DT_182001_A028_2025", "DT_1B040A5"]

for tbl_id in HARD_TABLES:
    row = catalog[catalog["tbl_id"] == tbl_id].iloc[0].to_dict()
    sample = pd.read_csv(raw_file(SAMPLE_DATA, row), dtype=str)
    cols = [c for c in sample.columns if c.endswith("_NM") and c != "TBL_NM"] + ["PRD_DE", "DT"]
    display(HTML(f"<h4>{escape(row['tbl_nm'])} <small>({row['category']} · {tbl_id} · {len(sample)}행)</small> "
                 f"— {table_link(tbl_id)}</h4>"))
    display(sample[cols])

,C1_OBJ_NM,C1_NM,C2_OBJ_NM,C2_NM,ITM_NM,UNIT_NM,PRD_DE,DT
0,특성별,전체,결혼 여부,사례수,비율,명,2025,2434
1,특성별,전체,결혼 여부,지금 배우자와 함께 살고 있다,비율,%,2025,56.2
2,특성별,전체,결혼 여부,지금은 혼자 살지만 예전에 결혼한 적이 있다(배우자가 돌아가셨거나 이혼하신 경우),비율,%,2025,34.3
3,특성별,전체,결혼 여부,지금까지 결혼한 적이 없다,비율,%,2025,5.5
4,특성별,전체,결혼 여부,배우자와 따로 살고 있다(예: 별거 중),비율,%,2025,1.3
5,특성별,전체,결혼 여부,기타(예: 동거 중 설명이 어려운 경우 등),비율,%,2025,1
6,특성별,전체,결혼 여부,잘 모르겠다,비율,%,2025,1.7
7,특성별,남성,결혼 여부,사례수,비율,명,2025,830
8,특성별,남성,결혼 여부,지금 배우자와 함께 살고 있다,비율,%,2025,79.9
9,특성별,남성,결혼 여부,지금은 혼자 살지만 예전에 결혼한 적이 있다(배우자가 돌아가셨거나 이혼하신 경우),비율,%,2025,12.7


,C1_OBJ_NM,C1_NM,C2_OBJ_NM,C2_NM,C3_OBJ_NM,C3_NM,ITM_NM,UNIT_NM,PRD_DE,DT
0,국적(지역)별,가나,성별,여자,체류자격별,합계,국적 지역 및 체류자격별 체류외국인 현황,명,2012,81
1,국적(지역)별,가나,성별,여자,체류자격별,외교(A-1),국적 지역 및 체류자격별 체류외국인 현황,명,2012,3
2,국적(지역)별,가나,성별,여자,체류자격별,공무(A-2),국적 지역 및 체류자격별 체류외국인 현황,명,2012,0
3,국적(지역)별,가나,성별,여자,체류자격별,협정(A-3),국적 지역 및 체류자격별 체류외국인 현황,명,2012,3
4,국적(지역)별,가나,성별,여자,체류자격별,미군현역(A-3-1),국적 지역 및 체류자격별 체류외국인 현황,명,2012,2
5,국적(지역)별,가나,성별,여자,체류자격별,미군군속(A-3-2),국적 지역 및 체류자격별 체류외국인 현황,명,2012,1
6,국적(지역)별,가나,성별,여자,체류자격별,기타협정(A-3-99),국적 지역 및 체류자격별 체류외국인 현황,명,2012,0
7,국적(지역)별,가나,성별,여자,체류자격별,사증면제(B-1),국적 지역 및 체류자격별 체류외국인 현황,명,2012,0
8,국적(지역)별,가나,성별,여자,체류자격별,관광통과(B-2),국적 지역 및 체류자격별 체류외국인 현황,명,2012,1
9,국적(지역)별,가나,성별,여자,체류자격별,일반무사증(B-2-1),국적 지역 및 체류자격별 체류외국인 현황,명,2012,1


In [109]:
# 표별 RAG에 있는 질문(y) · 없는 질문(n) — 골든셋 형식 (6·7단계 / 앱 JSON 붙여넣기와 같은 형식)
HARD_GOLDEN = [
    # ── 「결혼 여부」 ──
    {"question": "2025년 영주귀국 사할린동포 여성 중 지금 배우자와 함께 살고 있는 사람의 비율은?",
     "answer": "43.9%", "in_rag": "y",
     "link": "https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A028_2025",
     "note": "샘플 15행 (특성별 여성 · 지금 배우자와 함께 살고 있다). 남성 79.9%, 전체 56.2%. 비슷한 사할린 표와 헷갈리는지, 성별 조건을 적용하는지 확인"},
    {"question": "2025년 영주귀국 사할린동포 1세대 중 지금 배우자와 함께 살고 있는 사람의 비율은?",
     "answer": "확인 불가 (샘플 데이터에는 특성별 전체·남성·여성만 있음)", "in_rag": "n",
     "link": "https://kosis.kr/statHtml/statHtml.do?orgId=182&tblId=DT_182001_A028_2025",
     "note": "표는 RAG에 있지만 1세대 행은 샘플 20행 밖. 원본 전체 데이터 값은 48.3%. 전체 값(56.2%)을 1세대 값처럼 답하면 오답"},
    # ── 「국적(지역) 및 체류자격별 체류외국인 현황」 ──
    {"question": "2012년 가나 국적 여성 체류외국인 중 단기방문(C-3) 자격은 몇 명인가요?",
     "answer": "13명", "in_rag": "y",
     "link": "https://kosis.kr/statHtml/statHtml.do?orgId=111&tblId=DT_1B040A5",
     "note": "샘플 13행 (가나 · 여자 · 단기방문(C-3), 2012). 같은 이름 표 4개 중 2012년 표를 고르는지, 국적·체류자격 조건 없이도 값을 찾는지 확인. 가나 여성 전체는 81명"},
    {"question": "2012년 우리나라 체류외국인은 모두 몇 명인가요?",
     "answer": "확인 불가 (샘플 데이터에는 가나 여성 행만 있음)", "in_rag": "n",
     "link": "https://kosis.kr/statHtml/statHtml.do?orgId=111&tblId=DT_1B040A5",
     "note": "표는 RAG에 있지만 총계 행은 샘플 20행 밖. 원본 전체 데이터 값은 1,445,103명. 가나 여성 81명을 전체 값처럼 답하면 오답"},
]

display(pd.DataFrame(HARD_GOLDEN)[["in_rag", "question", "answer", "note"]])

# 골든셋에 추가하려면 아래 줄의 주석을 풀고 실행 (AI 문항 · 정답 검수 전으로 저장, 이미 있으면 건너뜀)
# add_golden(HARD_GOLDEN, author="AI")

,in_rag,question,answer,note
0,y,2025년 영주귀국 사할린동포 여성 중 지금 배우자와 함께 살고 있는 사람의 비율은?,43.9%,"샘플 15행 (특성별 여성 · 지금 배우자와 함께 살고 있다). 남성 79.9%, 전체 56.2%. 비슷한 사할린 표와 헷갈리는지, 성별..."
1,n,2025년 영주귀국 사할린동포 1세대 중 지금 배우자와 함께 살고 있는 사람의 비율은?,확인 불가 (샘플 데이터에는 특성별 전체·남성·여성만 있음),표는 RAG에 있지만 1세대 행은 샘플 20행 밖. 원본 전체 데이터 값은 48.3%. 전체 값(56.2%)을 1세대 값처럼 답하면 오답
2,y,2012년 가나 국적 여성 체류외국인 중 단기방문(C-3) 자격은 몇 명인가요?,13명,"샘플 13행 (가나 · 여자 · 단기방문(C-3), 2012). 같은 이름 표 4개 중 2012년 표를 고르는지, 국적·체류자격 조건 없..."
3,n,2012년 우리나라 체류외국인은 모두 몇 명인가요?,확인 불가 (샘플 데이터에는 가나 여성 행만 있음),"표는 RAG에 있지만 총계 행은 샘플 20행 밖. 원본 전체 데이터 값은 1,445,103명. 가나 여성 81명을 전체 값처럼 답하면 오답"
